In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [2]:
df_dk = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/dk_2007_2015_bn.csv")

In [3]:
df_dk = df_dk.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_dk

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,ROL,GE,COC,VAA,CPI,EUPC
0,3.69,6.42,6.45,2.96,137.0,0.44,22.06,25.231368,24.277939,0.540322,...,73.057143,0.00,1099.00,-4900.0,2.00,2.35,2.43,1.48,94.0,1.866582e+06
1,0.62,6.23,6.44,2.71,137.0,0.59,22.61,24.035733,24.986449,-1.000559,...,73.057143,-0.01,-207.00,-3557.0,1.95,2.24,2.38,1.54,93.0,2.784574e+06
2,1.17,6.06,6.43,2.76,138.0,0.54,19.85,19.032626,27.815694,-5.481583,...,73.100000,-0.01,-25.80,-2932.0,1.92,2.23,2.44,1.54,93.0,6.041840e+06
3,-3.57,6.17,6.37,2.54,139.0,0.44,19.37,17.822155,27.336108,1.132669,...,72.000000,-0.02,2364.18,251.0,1.90,2.10,2.35,1.54,93.0,7.495645e+06
4,3.94,6.29,6.32,2.39,139.0,0.41,19.85,18.987495,26.540559,0.894526,...,71.400000,0.00,1755.68,-5488.0,1.92,2.10,2.39,1.55,94.0,1.699311e+07
5,-5.00,5.73,5.97,2.64,140.0,0.38,19.85,19.226972,26.500323,-0.380977,...,71.100000,-0.01,656.38,-5127.0,1.88,1.99,2.38,1.67,90.0,6.354829e+06
6,0.20,5.45,5.64,2.59,140.0,0.42,19.73,19.352515,25.928504,0.970846,...,71.200000,-0.03,1134.28,1404.0,1.90,1.94,2.40,1.67,91.0,8.319061e+06
7,1.86,5.43,5.56,2.48,141.0,0.51,19.40,20.148203,25.797420,0.765766,...,71.000000,0.02,2013.96,-4198.0,2.10,1.78,2.25,1.52,92.0,2.383500e+07
8,0.61,5.59,5.58,2.49,142.0,0.71,19.56,20.543531,25.546356,1.385666,...,71.500000,0.03,950.39,-2024.0,2.02,1.79,2.19,1.55,91.0,1.274529e+07


In [5]:
sm = StructureModel()

In [6]:
sm.add_edges_from([
    ('EIAP', 'EUPC'),
])

In [7]:
sm.edges

OutEdgeView([('EIAP', 'EUPC')])

In [8]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2007_2015.html")

Graphs/fully_connected_dk_2007_2015.html


In [9]:
bn = BayesianNetwork(sm)

In [10]:
discretised_dk = pd.DataFrame(index=df_dk.index)

for col in df_dk.columns:
    no_unique = df_dk[col].nunique()
    
    if no_unique <= 1:
        discretised_dk[col] = 0
    else:
        try:
            discretised_dk[col] = pd.qcut(
                df_dk[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_dk[col] = df_dk[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_dk)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    2    2    2     2     0    1     2     2     0      1  ...    2    1   
1    1    2    2     2     0    2     2     2     0      0  ...    2    0   
2    1    1    2     2     0    2     1     0     2      0  ...    2    1   
3    0    1    1     1     1    1     0     0     2      2  ...    1    0   
4    2    2    1     0     1    0     1     0     2      1  ...    1    1   
5    0    1    1     1     1    0     1     1     1      0  ...    0    0   
6    0    0    0     1     1    0     1     1     1      2  ...    0    0   
7    2    0    0     0     2    1     0     1     1      1  ...    0    2   
8    1    0    0     0     2    2     0     2     0      2  ...    1    2   

   EPFRG  GGAG  ROL  GE  COC  VAA  CPI  EUPC  
0      1     0    2   2    2    0    2     0  
1      0     1    1   2    1    0    1     0  
2      0     1    1   2    2    0    1     0  
3      2     2    0   1

In [11]:
discretised_dk = discretised_dk.reset_index(drop=True)
bn.fit_node_states(discretised_dk)
baseline_auc = utils.get_avg_auc_all_info(discretised_dk, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 3.174290895462036 seconds
Processing fold 1 using 7 cores takes 2.9273900985717773 seconds
Processing fold 2 using 7 cores takes 3.077071189880371 seconds
Processing fold 3 using 7 cores takes 2.9864799976348877 seconds
Processing fold 4 using 7 cores takes 2.9246208667755127 seconds
Baseline AUC: 1.0


In [12]:
edges_to_add = [('LV', 'EIAP'), ('LV', 'EUPC')]
edges_to_remove = [('EIAP', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [13]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_dk_2007_2015.html")

Graphs/node_added_dk_2007_2015.html


In [14]:
discretised_dk['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_dk, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EIAP' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.091362714767456 seconds
Processing fold 1 using 7 cores takes 3.1050198078155518 seconds
Processing fold 2 using 7 cores takes 3.054720163345337 seconds
Processing fold 3 using 7 cores takes 3.039699077606201 seconds
Processing fold 4 using 7 cores takes 2.9671008586883545 seconds
AUC from adding LV between 'EIAP' and 'EUPC': 1.0
